### Unsloth


In [ ]:
import os
import json

# Load configuration
config_path = "../configs/hyperparameters.json"
with open(config_path, "r") as f:
    config = json.load(f)

os.environ["CUDA_VISIBLE_DEVICES"] = config["environment"]["cuda_visible_devices"]


In [ ]:
from unsloth import FastLanguageModel
import torch
from dotenv import load_dotenv

load_dotenv()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config["model"]["model_name"],
    max_seq_length = config["model"]["max_seq_length"],
    full_finetuning = config["model"]["full_finetuning"],
    token = os.getenv("HF_TOKEN")
)

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = config["dataset"]["chat_template"],
)

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("json", data_files=config["dataset"]["train_file"]).shuffle(seed=42)

# By default, it loads into a DatasetDict under the "train" split
print(dataset)

We now convert the reasoning dataset into conversational format:

In [ ]:
import penman

def remove_wiki(amr_text):
    triples = penman.decode(amr_text).triples
    triples = [x for x in triples if x[1] != ":wiki"]
    return penman.encode(penman.Graph(triples))

def flat_amr(amr_text):
    graph = penman.decode(amr_text)
    return penman.encode(graph, indent=0, compact=True).replace("\n", " ")

# Test the function
sample = dataset["train"][1]["amr"]
print("Original: \n",sample)
amr_removed_wiki = remove_wiki(sample)
print("Removed wiki: \n",amr_removed_wiki)
amr_flated = flat_amr(sample)
print("Flated: \n",amr_flated)

In [ ]:
def generate_conversation(examples):
    inputs  = examples["sentence"]
    reasonings = examples["reasoning"]
    ouputs = examples["amr"]
    system_prompts = examples["system_prompt"]
    
    conversations = []
    for input_text, reasoning, output, system_prompt in zip(inputs, reasonings, ouputs, system_prompts):
        output = remove_wiki(output)
        output = flat_amr(output)
        enable_thinking = config["inference"]["enable_thinking"]
        if enable_thinking:
            conversations.append([
                {"role" : "system",     "content" : system_prompt},
                {"role" : "user",      "content" : f"Convert the following English sentence into its Abstract Meaning Representation (AMR):\n\n<sentence>{input_text}</sentence>"},
                {"role" : "assistant", "content" : f"<think>\n{reasoning}\n</think>\n\n<amr>\n{output}\n</amr>"},
            ])
        else:
            conversations.append([
                {"role" : "system",     "content" : system_prompt},
                {"role" : "user",      "content" : f"Convert the following English sentence into its Abstract Meaning Representation (AMR):\n\n<sentence>{input_text}</sentence>"},
                {"role" : "assistant", "content" : f"<think>\n\n</think>\n\n<amr>\n{output}\n</amr>"},
            ])
            
    return { "conversations": conversations, }

# Deduplicate input sentences
seen_texts = set()
def filter_duplicates(example):
    if example["sentence"] in seen_texts:
        return False
    seen_texts.add(example["sentence"])
    return True

dataset = dataset.filter(filter_duplicates)
dataset = dataset.map(generate_conversation, batched = True)

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [ ]:
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
   return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# Phân tích phân phối số token trong dataset
import numpy as np
import matplotlib.pyplot as plt

def count_tokens(examples):
    texts = examples["text"]
    token_counts = []
    for text in texts:
        tokens = tokenizer(text, truncation=False)["input_ids"]
        token_counts.append(len(tokens))
    return {"token_count": token_counts}

# Áp dụng hàm đếm token
dataset_with_counts = dataset["train"].map(count_tokens, batched=True)

# Lấy danh sách số token
token_counts = dataset_with_counts["token_count"]

# Thống kê cơ bản
print(f"Tổng số mẫu: {len(token_counts)}")
print(f"Số token trung bình: {np.mean(token_counts):.2f}")
print(f"Số token trung vị: {np.median(token_counts):.2f}")
print(f"Số token min: {np.min(token_counts)}")
print(f"Số token max: {np.max(token_counts)}")
print(f"Độ lệch chuẩn: {np.std(token_counts):.2f}")
print(f"\nPercentiles:")
print(f"25%: {np.percentile(token_counts, 25):.0f}")
print(f"50%: {np.percentile(token_counts, 50):.0f}")
print(f"75%: {np.percentile(token_counts, 75):.0f}")
print(f"90%: {np.percentile(token_counts, 90):.0f}")
print(f"95%: {np.percentile(token_counts, 95):.0f}")
print(f"99%: {np.percentile(token_counts, 99):.0f}")

# Vẽ histogram
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(token_counts, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Số token')
plt.ylabel('Số lượng mẫu')
plt.title('Phân phối số token trong dataset')
plt.axvline(np.mean(token_counts), color='r', linestyle='--', label=f'Mean: {np.mean(token_counts):.0f}')
plt.axvline(np.median(token_counts), color='g', linestyle='--', label=f'Median: {np.median(token_counts):.0f}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(token_counts, vert=True)
plt.ylabel('Số token')
plt.title('Box plot phân phối số token')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Let's see how the chat template did!


In [ ]:
dataset["train"][100]['text']

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
config['model']['model_name'].split("/")[-1].lower()

In [ ]:
import os
import wandb
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d-%H%M")
tempt= config['model']['model_name'].split("/")[-1].lower()
run_name = f"{tempt}-amr-{timestamp}"
api_key = os.getenv("WANDB_API_KEY")
wandb.login(key=api_key)

run = wandb.init(
    project=f"sft-llm-amr-parsing", 
    name=run_name
)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_config = config["training"]

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = training_config["dataset_text_field"],
        packing = training_config["packing"],
        per_device_train_batch_size = training_config["per_device_train_batch_size"],
        gradient_accumulation_steps = training_config["gradient_accumulation_steps"],
        warmup_ratio = training_config["warmup_ratio"],
        num_train_epochs = training_config["num_train_epochs"],
        learning_rate = training_config["learning_rate"],
        logging_steps = training_config["logging_steps"],
        optim = training_config["optim"],
        weight_decay = training_config["weight_decay"],
        lr_scheduler_type = training_config["lr_scheduler_type"],
        report_to = training_config["report_to"],
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only

response_config = config["train_on_responses"]

trainer = train_on_responses_only(
    trainer,
    instruction_part = response_config["instruction_part"],
    response_part = response_config["response_part"],
)

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for instruct inference are `temperature = 0.7, top_p = 0.8, top_k = 20`

For reasoning chat based inference, `temperature = 0.6, top_p = 0.95, top_k = 20`

In [ ]:
input_text = "The cat sits on the mat."

messages = [
    {"role" : "user", "content" : f"Convert the following English sentence into its Abstract Meaning Representation (AMR):\n\n<sentence>{input_text}</sentence>"}
]

inference_config = config["inference"]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    # add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = inference_config["max_new_tokens"],
    temperature = inference_config["temperature"],
    top_p = inference_config["top_p"],
    top_k = inference_config["top_k"],
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:

# Push full model lên HF Hub
model_id = f"viamr-project/{run_name}"
model.push_to_hub(model_id, token = os.getenv("HF_TOKEN"))
tokenizer.push_to_hub(model_id, token = os.getenv("HF_TOKEN"))
from huggingface_hub import HfApi
import os

api = HfApi(token=os.getenv("HF_TOKEN"))
api.upload_file(
    path_or_fileobj=config_path,
    path_in_repo=os.path.basename(config_path),
    repo_id=model_id,
    token=os.getenv("HF_TOKEN")
)


print(f"✓ Full model đã được push lên: {run_name}")
print("Model này có thể dùng trực tiếp với vLLM!")

In [ ]:
# Giải phóng VRAM
import gc
import torch

# Xóa các object lớn (model, trainer)
if 'trainer' in dir():
    del trainer
if 'model' in dir():
    del model


# Chạy garbage collection
gc.collect()

# Xóa cache CUDA
torch.cuda.empty_cache()

# Kiểm tra VRAM đã được giải phóng
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

## Inference with vLLM

Now let's use vLLM for faster inference and evaluation on the test set.


In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from vllm import LLM, SamplingParams
from tqdm import tqdm

os.environ["CUDA_VISIBLE_DEVICES"] = config["environment"]["cuda_visible_devices"]

# Create benchmarks directory
benchmarks_dir = Path("benchmarks")
benchmarks_dir.mkdir(exist_ok=True)

# Model name for the benchmark file
model_name_safe = model_id.replace("/", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
benchmark_file = benchmarks_dir / f"{model_name_safe}.jsonl"

print(f"Benchmark results will be saved to: {benchmark_file}")

In [ ]:
# Initialize vLLM model
print(f"Loading model with vLLM: {model_id}")
print("This may take a few minutes...")

# Kiểm tra CUDA trước
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name()}")

vllm_config = config["vllm"]

try:
    vllm_model = LLM(
        model=model_id,
        tokenizer=model_id,
        tensor_parallel_size=vllm_config["tensor_parallel_size"],
        gpu_memory_utilization=vllm_config["gpu_memory_utilization"],
        max_model_len=vllm_config["max_model_len"],
        trust_remote_code=vllm_config["trust_remote_code"],
        dtype=vllm_config["dtype"],
        hf_token=os.getenv("HF_TOKEN"),
        enforce_eager=vllm_config["enforce_eager"],
    )
    print(f"✓ vLLM model loaded successfully: {model_id}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Configure sampling parameters (following Qwen3 recommendations)
vllm_sampling_config = config["vllm_sampling"]

sampling_params = SamplingParams(
    temperature=vllm_sampling_config["temperature"],
    top_p=vllm_sampling_config["top_p"],
    top_k=vllm_sampling_config["top_k"],
    max_tokens=vllm_sampling_config["max_tokens"],
    stop_token_ids=[tokenizer.eos_token_id],
)

print("Sampling parameters configured")
print(f"Temperature: {sampling_params.temperature}")
print(f"Top-p: {sampling_params.top_p}")
print(f"Top-k: {sampling_params.top_k}")
print(f"Max tokens: {sampling_params.max_tokens}")

In [ ]:
# Get test dataset
test_dataset = dataset["test"]
print(f"Test dataset size: {len(test_dataset)}")
print(f"Test dataset columns: {test_dataset.column_names}")

# Show first example
if len(test_dataset) > 0:
    print(f"\nFirst test example:")
    print(f"Input: {test_dataset[0]['sentence']}")
    print(f"Expected AMR: {test_dataset[0]['amr']}")


In [ ]:
prompts = []
for example in test_dataset:
    input_text = example["sentence"]
    
    # Create chat message
    messages = [
        {"role": "user", "content": f"Convert the following English sentence into its Abstract Meaning Representation (AMR):\n\n<sentence>{input_text}</sentence>"}
    ]
    
    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=config["inference"]["enable_thinking"]
    )
    
    prompts.append(prompt)

print(f"Prepared {len(prompts)} prompts for inference")
print(f"\nExample prompt:\n{prompts[0][:500]}...")

In [ ]:
# Run batch inference with vLLM
print(f"Starting inference on {len(prompts)} samples...")
print("This may take a while...")

outputs = vllm_model.generate(prompts, sampling_params)

print(f"Inference completed! Generated {len(outputs)} outputs")

In [ ]:
# benchmark_file = "sample.jsonl"

In [ ]:
# Process outputs and save to JSONL
print(f"Saving results to {benchmark_file}...")

with open(benchmark_file, 'w', encoding='utf-8') as f:
    for idx, (output, example) in enumerate(zip(outputs, test_dataset)):
        # Extract generated text
        generated_text = output.outputs[0].text
        
        # Create result record
        result = {
            "index": idx,
            "input": example["sentence"],
            "ground_truth": example["amr"],
            "generated_text": generated_text,
            "prompt": output.prompt,
            "model": model_id,
            "timestamp": datetime.now().isoformat(),
            "sampling_params": {
                "temperature": sampling_params.temperature,
                "top_p": sampling_params.top_p,
                "top_k": sampling_params.top_k,
                "max_tokens": sampling_params.max_tokens,
            }
        }
        
        # Write as JSON line
        f.write(json.dumps(result, ensure_ascii=False) + '\n')

print(f"✓ Results saved to {benchmark_file}")
print(f"Total samples: {len(outputs)}")

In [ ]:
from benchmark import AMRBenchmark

# Khởi tạo
benchmark = AMRBenchmark()

# Gọi hàm score với đường dẫn input
score_result = benchmark.score(benchmark_file)
print(score_result)

# Store score for later use
benchmark_score = score_result

In [ ]:
# Push model to Hugging Face Hub
from huggingface_hub import HfApi, login
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from dotenv import load_dotenv
import os

load_dotenv()

# 1. Login to Hugging Face (chạy lần đầu để authenticate)
# login()  # Sẽ yêu cầu token từ https://huggingface.co/settings/tokens

# 2. Đường dẫn model đã convert



# 3. Create README with benchmark score
readme_content = f"""# {model_id.split('/')[1]}

## Model Information
- **Timestamp**: {timestamp}

## Benchmark Results
- **Benchmark File**: {benchmark_file.name}
- **Score**: 

	- **F1**: {benchmark_score["main"]["F1"]["result"]}
	- **Precision**: {benchmark_score["main"]["Precision"]["result"]}
	- **Recall**: {benchmark_score["main"]["Recall"]["result"]}

## Usage
```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{model_id}", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("{model_id}", trust_remote_code=True)
```

Sample Prompt:
```text
{prompts[0]}
```

## Training Details
- Framework: veRL (Reinforcement Learning)
- Task: AMR Parsing
"""

# Save README locally first
readme_path = "README_temp.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)
print(f"✓ README created at: {readme_path}")


# 6. Push README file
api = HfApi()
api.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo="README.md",
    repo_id=model_id,
    token=os.getenv("HF_TOKEN")
)
print(f"✓ README pushed successfully")
print(f"✓ Model pushed successfully to: https://huggingface.co/{model_id}")

# Delete temp file
if os.path.exists(readme_path):
    os.remove(readme_path)
    print(f"✓ Temporary README file deleted: {readme_path}")